# 🔬 EMI-Simulation — Três Efeitos Mateus Institucionais
### Modelo Baseado em Agente para análise de concentração científica no Brasil

> **Referência:** Mattedi et al. (2025) — *O Efeito Mateus Institucional na ciência brasileira: uma análise da concentração espacial dos INCTs por meio de Modelagem Baseada em Agente (ABM)*

---

### Os 3 loops do Efeito Mateus:

| Loop | Equação | Mecanismo |
|------|---------|-----------|
| **EMI-I** Infraestrutura | `I(t+1) = I(t) + γI·R − δI·I(t)` | Capital físico → custo marginal menor |
| **EMI-T** Reputação | `T(t+1) = T(t) + γT·ln(1+R) − δT·T(t)` | Credibilidade cumulativa via publicações |
| **EMI-R** Recursos | `R ⊂ A(t) = φ[I+T+R](1−D) + ψ·PC` | Histórico financeiro → acesso direto |

**Equação de acesso (Eq. 01):**
$$A_i(t) = \varphi \cdot [I_i + T_i + R_i] \cdot (1-D_i) + \psi \cdot PC(t) \quad (\varphi + \psi = 1)$$

**Visibilidade científica (Eq. 02):**
$$V_i(t+1) = \eta_1 \cdot I_i(t+1) + \eta_2 \cdot T_i(t+1)$$

## 📦 1. Instalação de dependências

In [ ]:
!pip install numpy scipy matplotlib pandas -q
print("✅ Dependências instaladas")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from copy import deepcopy
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

CORES = {
    'Sudeste':      '#1a5276',
    'Sul':          '#2980b9',
    'Nordeste':     '#e67e22',
    'Centro-Oeste': '#27ae60',
    'Norte':        '#c0392b',
}
REGIOES = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sul', 'Sudeste']
print("✅ Imports OK")

## ⚙️ 2. Motor de Simulação

### Arquitetura dos 3 loops EMI

```
Recursos (R)
    │
    ├──[γI]──→  I(t+1) = I(t) + γI·R − δI·I(t)    ← EMI-I: Infraestrutura
    │
    ├──[γT]──→  T(t+1) = T(t) + γT·ln(1+R) − δT·T  ← EMI-T: Reputação
    │
    └──────→  R ∈ A(t) = φ[I+T+R](1−D) + ψ·PC      ← EMI-R: Recursos
                              ↑
                        A(t) → R(t+1) ∝ A   (fecha o loop)
```

In [ ]:
class Agent:
    """Grupo de Pesquisa — vetor de estado [I, T, R, D] em [0,1]."""
    def __init__(self, id, name, region, infraestrutura, qualificacao,
                 recursos_historicos, desigualdade):
        self.id   = id
        self.name = name
        self.region = region
        self.I = float(np.clip(infraestrutura,     0, 1))
        self.T = float(np.clip(qualificacao,        0, 1))
        self.R = float(np.clip(recursos_historicos, 0, 1))
        self.D = float(np.clip(desigualdade,        0, 1))
        self.acesso             = 0.0
        self.visibilidade       = 0.0
        self.recursos_recebidos = 0.0

    def clone(self):
        a = Agent(self.id, self.name, self.region,
                  self.I, self.T, self.R, self.D)
        return a


class EMIEngine:
    """
    Motor que implementa os 3 loops do Efeito Mateus Institucional.
    Cada loop pode ser ativado/desativado independentemente.
    """
    PERIFERICA   = {'Norte', 'Nordeste', 'Centro-Oeste'}
    CONSOLIDADA  = {'Sudeste', 'Sul'}

    def __init__(self, phi=0.7, psi=None,
                 gamma_i=0.18, delta_i=0.035,
                 gamma_t=0.11, delta_t=0.048,
                 eta_1=0.60,   eta_2=0.40,
                 zeta=1.0, pc=0.30,
                 use_emi_i=True, use_emi_t=True, use_emi_r=True,
                 psi_dynamic=False, psi_target=0.5, psi_steps=5,
                 fiscal_shock=None, shock_start=None):
        self.phi = phi
        self.psi = (1.0 - phi) if psi is None else psi
        self.gamma_i, self.delta_i = gamma_i, delta_i
        self.gamma_t, self.delta_t = gamma_t, delta_t
        self.eta_1,   self.eta_2   = eta_1, eta_2
        self.zeta = zeta
        self.pc   = pc
        self.use_i, self.use_t, self.use_r = use_emi_i, use_emi_t, use_emi_r
        self.psi_dynamic = psi_dynamic
        self.psi_target  = psi_target
        self.psi_steps   = psi_steps
        self.fiscal_shock = fiscal_shock   # multiplicador (ex: 0.7 = -30%)
        self.shock_start  = shock_start

    # ── Loop EMI-I ──────────────────────────────────────────────────────
    def _emi_i(self, agent, r_new):
        if not self.use_i: return agent.I
        return float(np.clip(agent.I + self.gamma_i * r_new - self.delta_i * agent.I, 0, 1))

    # ── Loop EMI-T ──────────────────────────────────────────────────────
    def _emi_t(self, agent, r_new):
        if not self.use_t: return agent.T
        return float(np.clip(agent.T + self.gamma_t * np.log(1+r_new) - self.delta_t * agent.T, 0, 1))

    # ── Loop EMI-R ──────────────────────────────────────────────────────
    def _emi_r(self, agent, r_new):
        if not self.use_r: return agent.R
        return float(np.clip(agent.R + 0.05 * r_new, 0, 1))

    # ── Acesso (Eq. 01) ─────────────────────────────────────────────────
    def access(self, agent, psi_ov=None):
        parts = ([agent.I] if self.use_i else []) +                     ([agent.T] if self.use_t else []) +                     ([agent.R] if self.use_r else [])
        merit = float(np.mean(parts)) if parts else 0.0
        psi   = psi_ov if psi_ov is not None else self.psi
        return self.phi * merit * (1 - agent.D) + psi * self.pc

    # ── Visibilidade (Eq. 02) ───────────────────────────────────────────
    def visibility(self, i, t):
        return self.eta_1 * i + self.eta_2 * t

    # ── Distribuição de recursos ────────────────────────────────────────
    def _distribute(self, agents, psi_ov=None):
        scores = [self.access(a, psi_ov) for a in agents]
        total  = sum(scores) or 1.0
        return [s / total for s in scores]

    def _coalition(self, agents, res):
        if self.zeta <= 1.0: return res
        if not any(a.region in self.CONSOLIDADA for a in agents): return res
        boosted = [r * self.zeta if a.region in self.PERIFERICA else r
                   for a, r in zip(agents, res)]
        total = sum(boosted) or 1.0
        return [b / total for b in boosted]

    # ── Passo t → t+1 ───────────────────────────────────────────────────
    def step(self, agents, psi_ov=None, fiscal_mult=1.0):
        for a in agents:
            a.acesso = self.access(a, psi_ov)
        res = self._distribute(agents, psi_ov)
        res = [r * fiscal_mult for r in res]
        total = sum(res) or 1.0
        res = [r / total for r in res]
        res = self._coalition(agents, res)
        for a, r in zip(agents, res):
            a.recursos_recebidos = r
            new_i = self._emi_i(a, r)
            new_t = self._emi_t(a, r)
            a.R   = self._emi_r(a, r)
            a.I   = new_i
            a.T   = new_t
            a.visibilidade = self.visibility(new_i, new_t)
        acc = [a.acesso for a in agents]
        return agents, {
            'gini':    _gini(acc),
            'shares':  _shares(agents),
            'emi_i':   np.mean([a.I for a in agents]),
            'emi_t':   np.mean([a.T for a in agents]),
            'emi_r':   np.mean([a.R for a in agents]),
        }

    # ── Simulação completa ───────────────────────────────────────────────
    def run(self, agents_dicts, timesteps=14, seed=42,
            inct_shocks=None, shock_mult=None, tol=0.01):
        np.random.seed(seed)
        agents = [Agent(**d) for d in agents_dicts]
        hist, prev_gini, psi_cur = [], None, self.psi
        for t in range(timesteps):
            if self.psi_dynamic and t <= self.psi_steps:
                psi_cur = self.psi + (self.psi_target - self.psi) * (t / self.psi_steps)
            if inct_shocks and t in inct_shocks:
                idx  = inct_shocks.index(t)
                mult = (shock_mult or [2.0, 1.8, 1.5])[idx]
                for a in agents:
                    if a.region in self.CONSOLIDADA:
                        a.I = float(np.clip(a.I * mult, 0, 1))
            fm = self.fiscal_shock if (self.fiscal_shock and self.shock_start
                                       and t >= self.shock_start) else 1.0
            agents, m = self.step(agents, psi_ov=psi_cur, fiscal_mult=fm)
            conv = abs(m['gini'] - prev_gini) if prev_gini is not None else 1.0
            hist.append({**m, 't': t, 'conv': conv})
            if t > 5 and conv < tol: break
            prev_gini = m['gini']
        sf = hist[-1]['shares']
        se = sf.get('Sudeste', 0)
        no = max(sf.get('Norte', 0.01), 0.01)
        return {
            'hist': hist,
            'agents': [vars(a) for a in agents],
            'gini_0':   hist[0]['gini'],
            'gini_f':   hist[-1]['gini'],
            'shares_f': sf,
            'SE_N':     round(se / no, 2),
            'ticks':    len(hist),
            'effects':  {'EMI-I': self.use_i, 'EMI-T': self.use_t, 'EMI-R': self.use_r},
        }

    def run_n(self, agents_dicts, timesteps=14, n=30, **kw):
        """N réplicas → média + IC 95% por tick."""
        all_gini, all_sh = {}, {r: {} for r in REGIOES}
        for seed in range(1, n + 1):
            res = self.run(deepcopy(agents_dicts), timesteps, seed=seed, **kw)
            for s in res['hist']:
                t = s['t']
                all_gini.setdefault(t, []).append(s['gini'])
                for r in REGIOES:
                    all_sh[r].setdefault(t, []).append(s['shares'].get(r, 0))
        ticks = sorted(all_gini)
        return {
            't':     ticks,
            'gini':  [np.mean(all_gini[t]) for t in ticks],
            'gini_lo': [np.mean(all_gini[t]) - 1.96*np.std(all_gini[t])/np.sqrt(n) for t in ticks],
            'gini_hi': [np.mean(all_gini[t]) + 1.96*np.std(all_gini[t])/np.sqrt(n) for t in ticks],
            'shares': {r: [np.mean(all_sh[r].get(t, [0])) for t in ticks] for r in REGIOES},
        }


# ── Helpers ─────────────────────────────────────────────────────────────────

def _gini(vals):
    arr = np.array(sorted(vals), dtype=float)
    if arr.sum() == 0: return 0.0
    n = len(arr); cs = np.cumsum(arr)
    return float((n + 1 - 2 * cs.sum() / cs[-1]) / n)

def _shares(agents):
    total = sum(a.acesso for a in agents) or 1.0
    return {r: round(sum(a.acesso for a in agents if a.region == r) / total * 100, 2)
            for r in REGIOES}

print("✅ EMIEngine carregado — 3 loops prontos: EMI-I | EMI-T | EMI-R")

## 📐 3. Parâmetros Calibrados e Agentes Padrão

In [ ]:
# ── Parâmetros calibrados (Tabela 2 do artigo) ──────────────────────────────
PARAMS = dict(
    phi=0.70, gamma_i=0.18, delta_i=0.035,
    gamma_t=0.11, delta_t=0.048, eta_1=0.60, eta_2=0.40,
    zeta=1.0, pc=0.30,
)

IC95 = {
    'gamma_i': (0.15, 0.21), 'delta_i': (0.030, 0.040),
    'gamma_t': (0.09, 0.13), 'delta_t': (0.040, 0.056),
    'eta_1':   (0.55, 0.65), 'eta_2':   (0.35, 0.45),
}

# ── Agentes padrão: principais universidades brasileiras ─────────────────────
AGENTES = [
    # Sudeste — consolidados
    dict(id='USP',    name='USP (SP)',    region='Sudeste',
         infraestrutura=0.95, qualificacao=0.92, recursos_historicos=0.90, desigualdade=0.10),
    dict(id='UNICAMP',name='UNICAMP(SP)', region='Sudeste',
         infraestrutura=0.88, qualificacao=0.87, recursos_historicos=0.85, desigualdade=0.10),
    dict(id='UFRJ',   name='UFRJ (RJ)',   region='Sudeste',
         infraestrutura=0.82, qualificacao=0.80, recursos_historicos=0.78, desigualdade=0.15),
    dict(id='UFMG',   name='UFMG (MG)',   region='Sudeste',
         infraestrutura=0.78, qualificacao=0.76, recursos_historicos=0.75, desigualdade=0.20),
    # Sul
    dict(id='UFRGS',  name='UFRGS(RS)',   region='Sul',
         infraestrutura=0.72, qualificacao=0.70, recursos_historicos=0.68, desigualdade=0.22),
    dict(id='UFSC',   name='UFSC (SC)',   region='Sul',
         infraestrutura=0.65, qualificacao=0.63, recursos_historicos=0.60, desigualdade=0.25),
    # Nordeste
    dict(id='UFPE',   name='UFPE (PE)',   region='Nordeste',
         infraestrutura=0.48, qualificacao=0.45, recursos_historicos=0.40, desigualdade=0.52),
    dict(id='UFC',    name='UFC  (CE)',   region='Nordeste',
         infraestrutura=0.45, qualificacao=0.42, recursos_historicos=0.38, desigualdade=0.55),
    dict(id='UFBA',   name='UFBA (BA)',   region='Nordeste',
         infraestrutura=0.42, qualificacao=0.40, recursos_historicos=0.35, desigualdade=0.58),
    # Centro-Oeste
    dict(id='UnB',    name='UnB  (DF)',   region='Centro-Oeste',
         infraestrutura=0.55, qualificacao=0.52, recursos_historicos=0.48, desigualdade=0.40),
    dict(id='UFG',    name='UFG  (GO)',   region='Centro-Oeste',
         infraestrutura=0.40, qualificacao=0.38, recursos_historicos=0.32, desigualdade=0.48),
    # Norte
    dict(id='UFPA',   name='UFPA (PA)',   region='Norte',
         infraestrutura=0.30, qualificacao=0.28, recursos_historicos=0.22, desigualdade=0.70),
    dict(id='UFAM',   name='UFAM (AM)',   region='Norte',
         infraestrutura=0.25, qualificacao=0.22, recursos_historicos=0.18, desigualdade=0.72),
]

df_ag = pd.DataFrame(AGENTES)[['name','region','infraestrutura','qualificacao',
                                'recursos_historicos','desigualdade']]
print(df_ag.to_string(index=False))

## 🚀 4. Simulações Básicas

In [ ]:
# EMI Pleno (phi=1, psi=0) — máxima concentração
eng_pleno = EMIEngine(phi=1.0, **{k: PARAMS[k] for k in PARAMS if k != 'phi'})
r_pleno   = eng_pleno.run(deepcopy(AGENTES), timesteps=14, seed=42)

print("=" * 55)
print("EMI PLENO  (phi=1.0, psi=0.0)  —  H0: politicas inertes")
print("=" * 55)
print(f"Gini inicial  : {r_pleno['gini_0']:.4f}")
print(f"Gini final    : {r_pleno['gini_f']:.4f}")
print(f"Razao SE/Norte: {r_pleno['SE_N']:.2f}x")
print()
print("Participacao regional final:")
for reg in REGIOES:
    bar = '█' * int(r_pleno['shares_f'].get(reg, 0) / 2)
    print(f"  {reg:15s}: {r_pleno['shares_f'].get(reg, 0):5.1f}%  {bar}")

print()
# EMI com psi=0.7 — patamar recomendado pelo artigo
eng_bal = EMIEngine(phi=0.3, **{k: PARAMS[k] for k in PARAMS if k != 'phi'})
r_bal   = eng_bal.run(deepcopy(AGENTES), timesteps=14, seed=42)
print("=" * 55)
print("EMI BALANCEADO  (phi=0.3, psi=0.7)  —  H1: politica ativa")
print("=" * 55)
print(f"Gini final    : {r_bal['gini_f']:.4f}  (queda de {r_pleno['gini_f'] - r_bal['gini_f']:.4f})")
print(f"Razao SE/Norte: {r_bal['SE_N']:.2f}x")
print()
print("Participacao regional final:")
for reg in REGIOES:
    bar = '█' * int(r_bal['shares_f'].get(reg, 0) / 2)
    print(f"  {reg:15s}: {r_bal['shares_f'].get(reg, 0):5.1f}%  {bar}")

## 🔬 5. Isolamento dos 3 Loops EMI

In [ ]:
configs = {
    'Somente EMI-I\n(Infraestrutura)':  dict(use_emi_i=True,  use_emi_t=False, use_emi_r=False),
    'Somente EMI-T\n(Reputacao)':       dict(use_emi_i=False, use_emi_t=True,  use_emi_r=False),
    'Somente EMI-R\n(Recursos)':        dict(use_emi_i=False, use_emi_t=False, use_emi_r=True),
    'EMI Completo\n(I+T+R)':            dict(use_emi_i=True,  use_emi_t=True,  use_emi_r=True),
}

resultados_iso = {}
for label, cfg in configs.items():
    eng = EMIEngine(**PARAMS, **cfg)
    resultados_iso[label] = eng.run(deepcopy(AGENTES), timesteps=14, seed=42)

labels  = list(resultados_iso.keys())
ginis_f = [resultados_iso[l]['gini_f'] for l in labels]
se_n    = [resultados_iso[l]['SE_N'] for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Isolamento dos 3 Efeitos Mateus Institucionais', fontsize=14, fontweight='bold')

# Gráfico 1 — Gini final
cores_iso = ['#2980b9', '#27ae60', '#e74c3c', '#8e44ad']
bars = axes[0].bar(labels, ginis_f, color=cores_iso, width=0.6, edgecolor='white')
axes[0].set_title('Coeficiente de Gini final por loop', fontsize=12)
axes[0].set_ylabel('Gini')
axes[0].set_ylim(0, max(ginis_f) * 1.25)
for bar, g in zip(bars, ginis_f):
    axes[0].text(bar.get_x() + bar.get_width()/2, g + 0.003, f'{g:.4f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].axhline(0.35, color='gray', ls='--', lw=1, label='âncora empírica (0.35)')
axes[0].legend(fontsize=9)

# Gráfico 2 — Razao SE/Norte
bars2 = axes[1].bar(labels, se_n, color=cores_iso, width=0.6, edgecolor='white')
axes[1].set_title('Razão Sudeste / Norte (acesso)', fontsize=12)
axes[1].set_ylabel('Razão SE / Norte')
axes[1].set_ylim(0, max(se_n) * 1.25)
for bar, s in zip(bars2, se_n):
    axes[1].text(bar.get_x() + bar.get_width()/2, s + 0.1, f'{s:.1f}x',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].axhline(5, color='gray', ls='--', lw=1, label='âncora empírica (~5:1)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('emi_loops_isolamento.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print("🏆 Efeito mais concentrador (isolado):",
      labels[ginis_f.index(max(ginis_f))].replace('\n', ' '))
print("✅ Efeito menos concentrador (isolado):",
      labels[ginis_f.index(min(ginis_f))].replace('\n', ' '))

## 🎞️ CÉLULA 12 — Mesa + Animação do EMI

Execute a célula de código imediatamente abaixo para gerar a animação da simulação EMI com uma camada Mesa.


In [ ]:

# ============================================================
# CÉLULA 12 — MESA + ANIMAÇÃO DO EMI
# ============================================================
# Requisitos prévios: as células anteriores já devem ter definido:
# AGENTES, PARAMS, REGIOES, CORES, _gini.

import sys, subprocess, importlib.util

# Instala Mesa somente se não estiver disponível no ambiente.
# No Google Colab/Jupyter com internet, isso resolve automaticamente.
if importlib.util.find_spec("mesa") is None:
    print("Mesa não encontrado. Instalando Mesa...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mesa", "-q"])

import mesa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from copy import deepcopy

# Aumenta o limite de incorporação HTML para evitar que a animação seja suprimida.
plt.rcParams["animation.embed_limit"] = 80


class EMIInstitutionAgent(mesa.Agent):
    """Agente institucional Mesa: universidade/grupo de pesquisa."""

    def __init__(self, model, data):
        # Compatibilidade Mesa 3.x/4.x e Mesa 2.x
        try:
            super().__init__(model)                 # Mesa 3.x+
        except TypeError:
            super().__init__(data["id"], model)     # Mesa 2.x

        self.code = data["id"]
        self.name = data["name"]
        self.region = data["region"]

        self.I = float(np.clip(data["infraestrutura"], 0, 1))
        self.T = float(np.clip(data["qualificacao"], 0, 1))
        self.R = float(np.clip(data["recursos_historicos"], 0, 1))
        self.D = float(np.clip(data["desigualdade"], 0, 1))

        self.acesso = 0.0
        self.visibilidade = 0.0
        self.recursos_recebidos = 0.0

    def as_record(self, tick):
        return {
            "tick": tick,
            "id": self.code,
            "name": self.name,
            "region": self.region,
            "I": self.I,
            "T": self.T,
            "R": self.R,
            "D": self.D,
            "acesso": self.acesso,
            "visibilidade": self.visibilidade,
            "recursos_recebidos": self.recursos_recebidos,
        }


class EMIMesaModel(mesa.Model):
    """Camada Mesa para simular e animar o Efeito Mateus Institucional."""

    PERIFERICA = {"Norte", "Nordeste", "Centro-Oeste"}
    CONSOLIDADA = {"Sudeste", "Sul"}

    def __init__(
        self,
        agents_dicts,
        params,
        phi=None,
        psi=None,
        zeta=None,
        seed=42,
        use_emi_i=True,
        use_emi_t=True,
        use_emi_r=True,
        fiscal_shock=None,
        shock_start=None,
    ):
        try:
            super().__init__(seed=seed)   # Mesa 3.x+
        except TypeError:
            super().__init__()            # Mesa 2.x

        self.phi = params.get("phi", 0.70) if phi is None else phi
        self.psi = (1.0 - self.phi) if psi is None else psi
        self.gamma_i = params.get("gamma_i", 0.18)
        self.delta_i = params.get("delta_i", 0.035)
        self.gamma_t = params.get("gamma_t", 0.11)
        self.delta_t = params.get("delta_t", 0.048)
        self.eta_1 = params.get("eta_1", 0.60)
        self.eta_2 = params.get("eta_2", 0.40)
        self.pc = params.get("pc", 0.30)
        self.zeta = params.get("zeta", 1.0) if zeta is None else zeta

        self.use_i = use_emi_i
        self.use_t = use_emi_t
        self.use_r = use_emi_r
        self.fiscal_shock = fiscal_shock
        self.shock_start = shock_start

        self.t = 0
        self.history = []
        self.snapshots = []

        self.institutions = [
            EMIInstitutionAgent(self, data)
            for data in deepcopy(agents_dicts)
        ]

        # Calcula acesso inicial antes do primeiro registro.
        for a in self.institutions:
            a.acesso = self._access(a)
            a.visibilidade = self._visibility(a.I, a.T)
        self._record_state()

    def _emi_i(self, agent, r_new):
        if not self.use_i:
            return agent.I
        return float(np.clip(agent.I + self.gamma_i * r_new - self.delta_i * agent.I, 0, 1))

    def _emi_t(self, agent, r_new):
        if not self.use_t:
            return agent.T
        return float(np.clip(agent.T + self.gamma_t * np.log(1 + r_new) - self.delta_t * agent.T, 0, 1))

    def _emi_r(self, agent, r_new):
        if not self.use_r:
            return agent.R
        return float(np.clip(agent.R + 0.05 * r_new, 0, 1))

    def _visibility(self, i, t):
        return self.eta_1 * i + self.eta_2 * t

    def _access(self, agent):
        parts = (
            ([agent.I] if self.use_i else [])
            + ([agent.T] if self.use_t else [])
            + ([agent.R] if self.use_r else [])
        )
        merit = float(np.mean(parts)) if parts else 0.0
        return self.phi * merit * (1 - agent.D) + self.psi * self.pc

    def _distribute_resources(self):
        for a in self.institutions:
            a.acesso = self._access(a)

        scores = np.array([a.acesso for a in self.institutions], dtype=float)
        resources = scores / (scores.sum() if scores.sum() > 0 else 1.0)

        if self.fiscal_shock is not None and self.shock_start is not None and self.t >= self.shock_start:
            resources = resources * float(self.fiscal_shock)
            resources = resources / (resources.sum() if resources.sum() > 0 else 1.0)

        if self.zeta > 1.0:
            boosted = np.array([
                r * self.zeta if a.region in self.PERIFERICA else r
                for a, r in zip(self.institutions, resources)
            ])
            resources = boosted / (boosted.sum() if boosted.sum() > 0 else 1.0)

        return resources

    def _regional_shares(self):
        total = sum(a.acesso for a in self.institutions) or 1.0
        return {
            r: round(sum(a.acesso for a in self.institutions if a.region == r) / total * 100, 2)
            for r in REGIOES
        }

    def _record_state(self):
        acc = [a.acesso for a in self.institutions]
        shares = self._regional_shares()
        self.history.append({
            "t": self.t,
            "gini": _gini(acc),
            "emi_i": np.mean([a.I for a in self.institutions]),
            "emi_t": np.mean([a.T for a in self.institutions]),
            "emi_r": np.mean([a.R for a in self.institutions]),
            **{f"share_{r}": shares.get(r, 0) for r in REGIOES},
        })
        self.snapshots.append(pd.DataFrame([a.as_record(self.t) for a in self.institutions]))

    def step(self):
        resources = self._distribute_resources()
        for a, r in zip(self.institutions, resources):
            a.recursos_recebidos = float(r)
            new_i = self._emi_i(a, r)
            new_t = self._emi_t(a, r)
            a.R = self._emi_r(a, r)
            a.I = new_i
            a.T = new_t
            a.visibilidade = self._visibility(new_i, new_t)

        self.t += 1
        self._record_state()


def rodar_mesa_emi(timesteps=14, phi=0.70, zeta=1.0, fiscal_shock=None, shock_start=None, seed=42):
    modelo = EMIMesaModel(
        AGENTES,
        PARAMS,
        phi=phi,
        zeta=zeta,
        fiscal_shock=fiscal_shock,
        shock_start=shock_start,
        seed=seed,
    )
    for _ in range(timesteps):
        modelo.step()
    return modelo


def animar_mesa_emi(modelo, intervalo_ms=850, salvar_html=True):
    """Gera e exibe animação HTML no próprio notebook."""
    snapshots = modelo.snapshots
    hist = pd.DataFrame(modelo.history)
    xmap = {reg: i for i, reg in enumerate(REGIOES)}

    max_acesso = max(df["acesso"].max() for df in snapshots)
    max_gini = max(hist["gini"].max(), 0.05)

    fig, (ax_agents, ax_gini) = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

    def update(frame):
        ax_agents.clear()
        ax_gini.clear()

        df = snapshots[frame].copy()
        tick = int(df["tick"].iloc[0])

        for reg in REGIOES:
            sub = df[df["region"] == reg].copy()
            if sub.empty:
                continue
            offsets = np.linspace(-0.18, 0.18, len(sub)) if len(sub) > 1 else [0]
            xs = np.array([xmap[reg] + off for off in offsets])
            ys = sub["acesso"].to_numpy()
            sizes = 130 + 2100 * sub["recursos_recebidos"].to_numpy()

            ax_agents.scatter(
                xs, ys, s=sizes,
                color=CORES[reg], alpha=0.78,
                edgecolor="white", linewidth=0.8,
                label=reg,
            )
            for x, y, nome in zip(xs, ys, sub["id"]):
                ax_agents.text(x, y + 0.006, nome, ha="center", va="bottom", fontsize=8)

        ax_agents.set_title(f"Agentes institucionais Mesa — tick {tick}")
        ax_agents.set_xticks(range(len(REGIOES)))
        ax_agents.set_xticklabels(REGIOES, rotation=20)
        ax_agents.set_ylabel("Acesso institucional")
        ax_agents.set_ylim(0, max_acesso * 1.25)
        ax_agents.grid(axis="y", alpha=0.25)

        hist_atual = hist[hist["t"] <= tick]
        ax_gini.plot(hist_atual["t"], hist_atual["gini"], lw=2, marker="o")
        ax_gini.set_title("Evolução da concentração — Gini")
        ax_gini.set_xlabel("Tick")
        ax_gini.set_ylabel("Gini")
        ax_gini.set_xlim(0, hist["t"].max())
        ax_gini.set_ylim(0, max_gini * 1.25)
        ax_gini.grid(alpha=0.25)

        g = float(hist.loc[hist["t"] == tick, "gini"].iloc[0])
        ax_gini.text(
            0.03, 0.92, f"Gini atual = {g:.4f}",
            transform=ax_gini.transAxes,
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85),
        )

        return ax_agents, ax_gini

    anim = FuncAnimation(fig, update, frames=len(snapshots), interval=intervalo_ms, repeat=True)
    html_obj = HTML(anim.to_jshtml())

    if salvar_html:
        with open("emi_mesa_animacao.html", "w", encoding="utf-8") as f:
            f.write(anim.to_jshtml())
        print("Arquivo HTML salvo no ambiente do notebook: emi_mesa_animacao.html")

    plt.close(fig)
    display(html_obj)
    return anim


# ============================================================
# EXECUÇÃO IMEDIATA DA ANIMAÇÃO
# ============================================================
modelo_mesa = rodar_mesa_emi(
    timesteps=14,
    phi=0.70,
    zeta=1.0,
    fiscal_shock=None,
    shock_start=None,
    seed=42,
)

animacao_emi = animar_mesa_emi(modelo_mesa, intervalo_ms=850, salvar_html=True)

# Tabela-resumo final da simulação Mesa
print("\nResumo final Mesa-EMI:")
display(pd.DataFrame(modelo_mesa.history).tail(1).T)


## 📊 6. Cenários Clássicos (artigo, §3.1)

In [ ]:
cenarios = {
    'EMI Pleno\n(phi=1, psi=0)':       dict(phi=1.0),
    'Bal. 60/40\n(phi=0.6, psi=0.4)':  dict(phi=0.6),
    'Bal. 40/60\n(phi=0.4, psi=0.6)':  dict(phi=0.4),
    'Coalizao\n(phi=0.5, psi=0.5,z=4)':dict(phi=0.5, zeta=4.0),
    'EMI Bloq.\n(phi=0, psi=1)':       dict(phi=0.0),
}

rows = []
for label, ov in cenarios.items():
    p = {**PARAMS, **ov}
    eng = EMIEngine(**p)
    res = eng.run(deepcopy(AGENTES), timesteps=14, seed=42)
    row = {'Cenário': label.replace('\n',' '), 'Gini': res['gini_f'], 'SE/N': res['SE_N']}
    row.update({r: res['shares_f'].get(r, 0) for r in REGIOES})
    rows.append(row)

df_cen = pd.DataFrame(rows)
print(df_cen.to_string(index=False, float_format='%.2f'))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cenários Clássicos — EMI (artigo, Quadro 2.3)', fontsize=14, fontweight='bold')

nomes  = [r['Cenário'] for r in rows]
x      = np.arange(len(nomes))
ginis  = [r['Gini'] for r in rows]

# Gráfico 1 — Gini
clrs = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(nomes)))
for i, (g, c) in enumerate(zip(ginis, clrs)):
    axes[0].bar(i, g, color=c, width=0.65, edgecolor='white')
    axes[0].text(i, g + 0.005, f'{g:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(nomes, fontsize=8)
axes[0].set_ylabel('Gini')
axes[0].set_title('Coeficiente de Gini por cenário')
axes[0].set_ylim(0, 0.45)
axes[0].axhline(0.35, ls='--', color='gray', lw=1, label='empírico 2022')
axes[0].legend(fontsize=9)

# Gráfico 2 — Barras empilhadas por região
bottoms = np.zeros(len(nomes))
for reg in REGIOES:
    vals = [r[reg] for r in rows]
    axes[1].bar(x, vals, bottom=bottoms, color=CORES[reg], label=reg, width=0.65, edgecolor='white')
    bottoms += np.array(vals)
axes[1].set_xticks(x)
axes[1].set_xticklabels(nomes, fontsize=8)
axes[1].set_ylabel('Participação (%)')
axes[1].set_title('Distribuição regional por cenário')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_ylim(0, 110)
axes[1].axhline(34, ls=':', color='white', lw=0.8, alpha=0.7)

plt.tight_layout()
plt.savefig('emi_cenarios_classicos.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏛️ 7. Simulação de Política Pública — Varredura de ψ × ζ

In [ ]:
psi_vals  = np.arange(0.0, 1.05, 0.1)
zeta_vals = [1.0, 2.0, 4.0]
exp_rows  = []

for z in zeta_vals:
    for psi in psi_vals:
        p = {**PARAMS, 'phi': round(1-psi, 2), 'zeta': z}
        eng = EMIEngine(**p)
        res = eng.run(deepcopy(AGENTES), timesteps=14, seed=42)
        exp_rows.append({
            'psi': round(psi, 2), 'zeta': z,
            'gini': res['gini_f'],
            'norte': res['shares_f'].get('Norte', 0),
            'nordeste': res['shares_f'].get('Nordeste', 0),
            'sudeste': res['shares_f'].get('Sudeste', 0),
        })

df_pol = pd.DataFrame(exp_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Varredura de politica compensatória (psi) e coalizão (zeta)', fontsize=13, fontweight='bold')

cores_z = {'1.0': '#e74c3c', '2.0': '#e67e22', '4.0': '#27ae60'}

for z in zeta_vals:
    sub = df_pol[df_pol['zeta'] == z]
    lbl = f'zeta={z}'
    axes[0].plot(sub['psi'], sub['gini'],    marker='o', ms=5, label=lbl, color=cores_z[str(z)])
    axes[1].plot(sub['psi'], sub['norte'],   marker='o', ms=5, label=lbl, color=cores_z[str(z)])
    axes[2].plot(sub['psi'], sub['sudeste'], marker='o', ms=5, label=lbl, color=cores_z[str(z)])

axes[0].set_title('Gini vs psi');     axes[0].set_xlabel('psi'); axes[0].set_ylabel('Gini')
axes[1].set_title('Norte% vs psi');   axes[1].set_xlabel('psi'); axes[1].set_ylabel('%')
axes[2].set_title('Sudeste% vs psi'); axes[2].set_xlabel('psi'); axes[2].set_ylabel('%')

for ax in axes:
    ax.legend(fontsize=9)
    ax.axvline(0.7, ls='--', color='gray', lw=1, alpha=0.7)  # limiar recomendado

# Anotação do psi_critico
axes[0].axhline(0.30, ls=':', color='#2c3e50', lw=1)
axes[0].text(0.02, 0.31, 'Gini = 0.30', fontsize=8, color='#2c3e50')
axes[0].text(0.71, axes[0].get_ylim()[1]*0.95, 'psi=0.7\n(recomendado)', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('emi_politica_psi_zeta.png', dpi=150, bbox_inches='tight')
plt.show()

# Identifica psi*
linha_z1 = df_pol[df_pol['zeta'] == 1.0].sort_values('psi')
psi_crit = linha_z1[linha_z1['gini'] < 0.30]['psi'].min()
print(f"psi_critico (zeta=1, Gini < 0.30): {psi_crit}")
print("→ Recomendacao do artigo: psi >= 0.7 (Mattedi et al., 2025, secao 4)")

# Heatmap Gini: psi x zeta
psi_v = np.round(np.arange(0, 1.1, 0.1), 1)
zeta_v = [1.0, 2.0, 3.0, 4.0, 6.0]
matrix = np.zeros((len(zeta_v), len(psi_v)))

for i, z in enumerate(zeta_v):
    for j, ps in enumerate(psi_v):
        p = {**PARAMS, 'phi': round(1-ps, 2), 'zeta': z}
        eng = EMIEngine(**p)
        res = eng.run(deepcopy(AGENTES), 14, seed=42)
        matrix[i, j] = res['gini_f']

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.40)
ax.set_xticks(range(len(psi_v)))
ax.set_xticklabels([f'{p:.1f}' for p in psi_v])
ax.set_yticks(range(len(zeta_v)))
ax.set_yticklabels([f'{z:.0f}' for z in zeta_v])
ax.set_xlabel('psi (peso compensatorio)')
ax.set_ylabel('zeta (bonus de coalizao)')
ax.set_title('Heatmap do Gini final: psi x zeta  (verde = mais equitativo)', fontsize=12)
for i in range(len(zeta_v)):
    for j in range(len(psi_v)):
        ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if matrix[i,j] > 0.20 else 'black')
plt.colorbar(im, ax=ax, label='Gini')
plt.tight_layout()
plt.savefig('emi_heatmap_psi_zeta.png', dpi=150, bbox_inches='tight')
plt.show()

## 💥 8. Cenários de Choque Fiscal (Quadro 7 do artigo)

In [ ]:
# Correção da célula 8:
# 1) remove troca explícita de backend dentro da célula;
# 2) evita `bbox_inches="tight"`, que pode gerar PNG com altura anômala em Colab/Jupyter;
# 3) separa Gini e participação regional em eixos Y distintos no gráfico C4.

import traceback

try:
    # ── C1: simulações antes da criação da figura ───────────────────────
    eng_base  = EMIEngine(**PARAMS)
    eng_shock = EMIEngine(**PARAMS, fiscal_shock=0.70, shock_start=3)

    res_base  = eng_base.run_n(deepcopy(AGENTES),  14, n=20)
    res_shock = eng_shock.run_n(deepcopy(AGENTES), 14, n=20)

    # ── C4: simulações antes da criação da figura ───────────────────────
    eng_din = EMIEngine(**{**PARAMS, 'phi': 1.0},
                        psi_dynamic=True, psi_target=0.5, psi_steps=5)
    res_din = eng_din.run_n(deepcopy(AGENTES), 14, n=20)

    eng_sta = EMIEngine(**{**PARAMS, 'phi': 1.0})
    res_sta = eng_sta.run_n(deepcopy(AGENTES), 14, n=20)

    # ── Conversão explícita para arrays: evita problemas em fill_between ─
    t_base  = np.asarray(res_base['t'], dtype=float)
    t_shock = np.asarray(res_shock['t'], dtype=float)
    t_sta   = np.asarray(res_sta['t'], dtype=float)
    t_din   = np.asarray(res_din['t'], dtype=float)

    # ── Cria figura depois de ter todos os dados ─────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    fig.suptitle('Cenários de Choque Fiscal (C1) e Política Dinâmica (C4)',
                 fontsize=13, fontweight='bold')

    # C1 — Gini com intervalo de confiança
    axes[0].plot(t_base, res_base['gini'],
                 label='Baseline (sem choque)', color='#2980b9', lw=2)
    axes[0].fill_between(t_base, res_base['gini_lo'], res_base['gini_hi'],
                         alpha=0.2, color='#2980b9')

    axes[0].plot(t_shock, res_shock['gini'],
                 label='C1: Choque -30% (tick 3)', color='#e74c3c', lw=2, ls='--')
    axes[0].fill_between(t_shock, res_shock['gini_lo'], res_shock['gini_hi'],
                         alpha=0.2, color='#e74c3c')

    axes[0].axvline(3, color='gray', ls=':', lw=1)
    axes[0].text(3.1, max(res_shock['gini']) * 0.9,
                 'choque', fontsize=9, color='gray')

    axes[0].set_title('C1 — Choque Fiscal CNPq (λ=0,70)')
    axes[0].set_xlabel('Tick')
    axes[0].set_ylabel('Gini')
    axes[0].set_ylim(0, max(max(res_base['gini_hi']), max(res_shock['gini_hi'])) * 1.15)
    axes[0].legend(fontsize=9)

    # C4 — eixo esquerdo: Gini
    axes[1].plot(t_sta, res_sta['gini'],
                 label='Gini: ψ=0 estático', color='#e74c3c', lw=2)
    axes[1].plot(t_din, res_din['gini'],
                 label='Gini: ψ 0→0,5 dinâmico', color='#27ae60', lw=2, ls='--')

    axes[1].set_title('C4 — Bônus Compensatório Dinâmico')
    axes[1].set_xlabel('Tick')
    axes[1].set_ylabel('Gini')
    axes[1].set_ylim(0, max(max(res_sta['gini']), max(res_din['gini'])) * 1.20)

    # C4 — eixo direito: participação regional (%)
    ax_share = axes[1].twinx()
    for reg in ['Norte', 'Nordeste']:
        ax_share.plot(t_din, res_din['shares'][reg],
                      label=f'{reg}: participação (%)',
                      ls=':', lw=1.8, color=CORES[reg])

    ax_share.set_ylabel('Participação regional (%)')
    ax_share.set_ylim(0, max(max(res_din['shares']['Norte']),
                             max(res_din['shares']['Nordeste'])) * 1.25)

    # Legenda combinada dos dois eixos
    lines_left, labels_left = axes[1].get_legend_handles_labels()
    lines_right, labels_right = ax_share.get_legend_handles_labels()
    axes[1].legend(lines_left + lines_right,
                   labels_left + labels_right,
                   fontsize=8, loc='best')

    # Não usar bbox_inches="tight" aqui: em alguns ambientes ele expande
    # indevidamente a área da figura e gera PNG vertical gigante.
    plt.savefig('emi_choques_fiscais.png', dpi=150)
    plt.show()
    plt.close(fig)

    print("C1 | Gini baseline:", round(res_base['gini'][-1], 4),
          "| Gini c/ choque:", round(res_shock['gini'][-1], 4),
          "| delta:", round(res_shock['gini'][-1] - res_base['gini'][-1], 4))

    print("C4 | Gini estático:", round(res_sta['gini'][-1], 4),
          "| Gini dinâmico:", round(res_din['gini'][-1], 4),
          "| Norte%:", round(res_din['shares']['Norte'][-1], 2))

except Exception:
    traceback.print_exc()

## 💰 9. Matching Funds — Contrapartida Federal Escalonada

In [ ]:
# Contrapartida inversamente proporcional ao PIB estadual
estados_exemplo = [
    {'uf': 'SP', 'pib_pc': 58000, 'regiao': 'Sudeste'},
    {'uf': 'RJ', 'pib_pc': 42000, 'regiao': 'Sudeste'},
    {'uf': 'MG', 'pib_pc': 32000, 'regiao': 'Sudeste'},
    {'uf': 'RS', 'pib_pc': 45000, 'regiao': 'Sul'},
    {'uf': 'SC', 'pib_pc': 40000, 'regiao': 'Sul'},
    {'uf': 'BA', 'pib_pc': 19000, 'regiao': 'Nordeste'},
    {'uf': 'PE', 'pib_pc': 18000, 'regiao': 'Nordeste'},
    {'uf': 'CE', 'pib_pc': 17000, 'regiao': 'Nordeste'},
    {'uf': 'DF', 'pib_pc': 82000, 'regiao': 'Centro-Oeste'},
    {'uf': 'GO', 'pib_pc': 28000, 'regiao': 'Centro-Oeste'},
    {'uf': 'PA', 'pib_pc': 16000, 'regiao': 'Norte'},
    {'uf': 'AM', 'pib_pc': 22000, 'regiao': 'Norte'},
]

def matching_funds(estados, budget, power=1.0):
    pibs = [e['pib_pc'] for e in estados]
    weights = [1.0 / (max(p, 1) ** power) for p in pibs]
    total_w = sum(weights)
    result = []
    for e, w in zip(estados, weights):
        result.append({**e, 'contrapartida': round(budget * w / total_w, 4),
                       'pct': round(w / total_w * 100, 2)})
    return sorted(result, key=lambda x: -x['contrapartida'])

budget_federal = 1.0  # R$ 1 bilhão disponível

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Matching Funds — Contrapartida Federal Inversamente Proporcional ao PIB',
             fontsize=13, fontweight='bold')

for ax, power, title in zip(axes, [1.0, 2.0],
                             ['Linear (p=1)', 'Quadrático (p=2) — favorece mais periferias']):
    res_mf = matching_funds(estados_exemplo, budget_federal, power)
    ufs    = [r['uf'] for r in res_mf]
    vals   = [r['contrapartida'] for r in res_mf]
    cores_mf = [CORES.get(r['regiao'], '#7f8c8d') for r in res_mf]
    bars = ax.bar(ufs, vals, color=cores_mf, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Contrapartida (R$ bi)')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
    # Legenda por regiao
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=CORES[r], label=r) for r in REGIOES if any(
               e['regiao'] == r for e in estados_exemplo)]
    ax.legend(handles=handles, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('emi_matching_funds.png', dpi=150, bbox_inches='tight')
plt.show()

## 📈 10. Evolução Temporal — Gini e Participação Regional

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8), constrained_layout=True)
fig.suptitle('Evolução Temporal do EMI — Cinco Cenários (30 réplicas, IC 95%)',
             fontsize=14, fontweight='bold')

cenarios_evo = {
    'EMI Pleno\n(phi=1, ψ=0)':          dict(phi=1.0),
    'Balanceado\n(phi=0.4, ψ=0.6)':    dict(phi=0.4),
    'Coalizão\n(phi=0.5, ζ=4)':        dict(phi=0.5, zeta=4.0),
    'Alta Compens.\n(phi=0.2, ψ=0.8)': dict(phi=0.2),
    'EMI Bloqueado\n(phi=0, ψ=1)':     dict(phi=0.0),
}

for col, (label, ov) in enumerate(cenarios_evo.items()):
    p = {**PARAMS, **ov}
    eng = EMIEngine(**p)
    res = eng.run_n(deepcopy(AGENTES), 14, n=30)
    t = np.asarray(res['t'], dtype=float)

    ax_top = axes[0, col]
    ax_bot = axes[1, col]

    # Linha superior: Gini
    ax_top.plot(t, res['gini'], color='#2c3e50', lw=2)
    ax_top.fill_between(t, res['gini_lo'], res['gini_hi'], alpha=0.25, color='#2c3e50')
    ax_top.set_title(label.replace('\n', ' '), fontsize=9)
    ax_top.set_ylabel('Gini')
    ax_top.set_ylim(0, 0.45)
    ax_top.axhline(0.35, ls='--', color='gray', lw=0.8)

    # Linha inferior: participação regional
    for reg in REGIOES:
        ax_bot.plot(t, res['shares'][reg], color=CORES[reg], label=reg, lw=1.5)
    ax_bot.set_ylabel('Participação (%)')
    ax_bot.set_xlabel('Tick')
    ax_bot.set_ylim(0, 65)
    if col == 0:
        ax_bot.legend(fontsize=7, loc='upper right')

plt.savefig('emi_evolucao_temporal.png', dpi=150)
plt.show()

## 🌐 11. API REST — Opcional (FastAPI + pyngrok)

> Execute as células abaixo se quiser expor a simulação como serviço HTTP com URL pública.
> Requer uma conta gratuita em [ngrok.com](https://ngrok.com) para obter o `NGROK_AUTH_TOKEN`.

In [ ]:
# Instala FastAPI, uvicorn e pyngrok
!pip install fastapi uvicorn pyngrok -q
print("✅ FastAPI + pyngrok instalados")

import os, threading, time, json
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

api = FastAPI(title='EMI-Simulation API', version='1.0.0')

class SimBody(BaseModel):
    phi:          float = 0.70
    psi_override: float | None = None
    zeta:         float = 1.0
    use_emi_i:    bool  = True
    use_emi_t:    bool  = True
    use_emi_r:    bool  = True
    timesteps:    int   = 14
    replications: int   = 10

@api.get('/')
def root():
    return {'api': 'EMI-Simulation', 'loops': ['EMI-I', 'EMI-T', 'EMI-R']}

@api.get('/emi/effects')
def effects():
    return {
        'EMI-I': 'I(t+1) = I(t) + gamma_I * R - delta_I * I(t)',
        'EMI-T': 'T(t+1) = T(t) + gamma_T * ln(1+R) - delta_T * T(t)',
        'EMI-R': 'R in A(t) = phi*[I+T+R]*(1-D) + psi*PC',
    }

@api.post('/simulation/run')
def run(body: SimBody):
    p = {**PARAMS, 'phi': body.phi, 'zeta': body.zeta,
         'use_emi_i': body.use_emi_i,
         'use_emi_t': body.use_emi_t,
         'use_emi_r': body.use_emi_r}
    eng = EMIEngine(**p)
    if body.replications > 1:
        res = eng.run_n(deepcopy(AGENTES), body.timesteps, n=body.replications)
    else:
        res = eng.run(deepcopy(AGENTES), body.timesteps)
    return JSONResponse(content=res)

@api.post('/emi/compare')
def compare(body: SimBody):
    out = {}
    cfgs = {
        'EMI-I': dict(use_emi_i=True,  use_emi_t=False, use_emi_r=False),
        'EMI-T': dict(use_emi_i=False, use_emi_t=True,  use_emi_r=False),
        'EMI-R': dict(use_emi_i=False, use_emi_t=False, use_emi_r=True),
        'ALL':   dict(use_emi_i=True,  use_emi_t=True,  use_emi_r=True),
    }
    for name, cfg in cfgs.items():
        p   = {**PARAMS, 'phi': body.phi, 'zeta': body.zeta, **cfg}
        eng = EMIEngine(**p)
        res = eng.run(deepcopy(AGENTES), body.timesteps)
        out[name] = {'gini_final': res['gini_f'], 'SE_N': res['SE_N'],
                     'shares': res['shares_f']}
    return JSONResponse(content=out)

# Inicia o servidor em thread separada
def _serve():
    uvicorn.run(api, host='0.0.0.0', port=8000, log_level='error')

t = threading.Thread(target=_serve, daemon=True)
t.start()
time.sleep(2)
print("✅ Servidor rodando em http://localhost:8000")
print("   Docs: http://localhost:8000/docs")

# Expõe via ngrok (requer NGROK_AUTH_TOKEN)
# Obtenha em: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = ''   # ← Cole seu token aqui

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000)
    print(f'🌐 URL pública: {tunnel.public_url}')
    print(f'📖 Docs:        {tunnel.public_url}/docs')
else:
    print('ℹ️  Configure NGROK_TOKEN para expor a API publicamente.')
    print('   Sem token: a API funciona apenas localmente (localhost:8000).')

---

## 📋 Resumo dos Resultados

| Métrica | EMI Pleno | Bal. (ψ=0.7) | EMI Bloqueado |
|---------|-----------|-------------|---------------|
| Gini final | ~0.38 | ~0.13 | 0.00 |
| Norte% | ~2.6% | ~11% | ~15% |
| Nordeste% | ~11% | ~19% | ~23% |
| Sudeste% | ~57% | ~42% | ~31% |

### Conclusões principais (Mattedi et al., 2025):
1. **EMI-R é o loop mais concentrador** — recursos históricos entram diretamente no acesso
2. **ψ ≥ 0.7 é condição necessária** para queda significativa do Gini
3. **ζ (coalizão) potencializa** a redistribuição, mas não substitui ψ
4. **Nordeste enfrenta saturação** por diluição (muitos grupos, FAP per capita baixo)
5. **H₀ rejeitada**: políticas compensatórias têm efeito positivo mensurável

### Recomendações de política (artigo, §4):
- Elevar ψ ≥ 0.7 nas próximas chamadas
- *Matching funds* escalonados inversamente ao PIB estadual
- Coliderança inter-regional obrigatória em consórcios
- Observatório de dados abertos para ajuste anual de φ e ψ